# 🔍 Critique approfondie — `train_classifier.py` et `train_regressor.py`

**Objectif de ce notebook** : comprendre en profondeur comment fonctionnent les deux modèles du pipeline SalesTeam AI, puis diagnostiquer précisément pourquoi le système peut produire des résultats absurdes en production — comme suggérer **300 unités d'un téléphone NOKIA** sans aucune justification visible.

Ce n'est pas une critique de style ou de propreté de code. Le code des deux fichiers est bien structuré, avec de bonnes pratiques (split anti-fuite par client, early stopping, comparaison contre baseline). Le problème est plus profond : il est **statistique et propre à la nature des données de vente**, pas un bug au sens classique.


---
## 1. Comment fonctionne `train_classifier.py` — en profondeur

### La question posée au modèle

`train_classifier.py` répond à une seule question binaire :

> *Pour cette paire (client, produit), va-t-il y avoir un achat le mois prochain ?*

### Le pipeline exact, étape par étape

```
1. Charger training_set.csv (153 320 lignes : positifs réels + négatifs échantillonnés)
2. Encoder categorie en nombre (LabelEncoder)
3. Séparer X (features) et y (target_bought)
4. Split train/val/test — PAR CLIENT (GroupShuffleSplit), jamais par ligne
5. Calculer scale_pos_weight sur le train uniquement (≈ 8.87 dans ce projet)
6. Entraîner XGBoost avec early stopping (max 500 arbres, arrêt si le PR-AUC
   de validation stagne pendant 30 arbres)
7. Évaluer : classification_report, ROC-AUC, PR-AUC, matrice de confusion
8. Sauvegarder modèle + encodeur + métadonnées JSON
```

### Le point le plus important à retenir

Le split **par client** (pas par ligne) est ce qui rend l'évaluation de ce modèle honnête. Sans lui, le modèle pourrait "mémoriser" un client via d'autres produits qu'il a déjà vus en entraînement, et le score affiché serait trompeusement optimiste. C'est un vrai bon réflexe déjà en place.

**Ce que ce modèle fait BIEN** : il isole la décision "acheter ou pas" de la décision "combien". C'est une architecture saine, comme un médecin qui diagnostique avant de doser un médicament.


---
## 2. Comment fonctionne `train_regressor.py` — en profondeur

### La question posée au modèle

`train_regressor.py` répond à une question numérique, mais **seulement pour les cas où le classifieur a déjà dit "oui"** :

> *Sachant que ce client va racheter ce produit, combien d'unités ?*

### Le pipeline exact, étape par étape

```
1. Charger training_set.csv PUIS FILTRER : ne garder que target_qty > 0
   (153 320 lignes -> ~15 536 lignes, les vrais achats uniquement)
2. Réutiliser EXACTEMENT le même LabelEncoder que le classifieur
   (jamais un nouveau fit — cohérence de l'encodage entre les 2 modèles)
3. Préparer X (features incluant avg_qty), y = target_qty, baseline = avg_qty
4. Split train/val/test PAR CLIENT (même logique anti-fuite)
5. Entraîner XGBoost Regressor :
      objective = "reg:squarederror"   (minimise l'erreur quadratique)
      eval_metric = "mae"              (mais on SURVEILLE le MAE, plus lisible)
      early_stopping_rounds = 30
6. Évaluer : MAE, RMSE, comparaison contre la baseline avg_qty
7. Clip (jamais négatif) + arrondi à l'entier + sauvegarde
```

### Pourquoi le filtrage sur `target_qty > 0` est essentiel

Le régresseur ne doit JAMAIS voir les négatifs — sinon il apprendrait à prédire 0 la plupart du temps, ce qui est déjà le travail du classifieur. C'est une séparation de responsabilités saine.

**Mais c'est exactement ce filtrage qui, combiné à d'autres facteurs qu'on va détailler, crée le problème du "300 unités".**


---
## 3. Le problème concret : "300 unités de NOKIA, sans explication"

Reproduisons exactement ce qui se passe en interne avec des chiffres réalistes, pour comprendre où naît ce genre de prédiction absurde.

### Cause n°1 — Une seule commande exceptionnelle peut dominer toute la statistique d'une paire

`avg_qty` est calculée par une simple moyenne arithmétique sur l'historique du client pour CE produit précis, sans aucun plafonnement (pas de winsorization, pas de clip de percentile). Une seule commande "bulk" (un revendeur qui restocke une fois, une commande groupée exceptionnelle, voire une erreur de saisie jamais corrigée) suffit à faire exploser la moyenne pour toujours.


In [1]:
import numpy as np

normal_orders = [4, 5, 3, 6, 4, 5]          # commandes NOKIA typiques de ce client
outlier_order = [300]                        # UNE commande exceptionnelle passée (restock, erreur...)

all_orders_with_outlier = normal_orders + outlier_order

avg_without = np.mean(normal_orders)
avg_with_outlier = np.mean(all_orders_with_outlier)
median_with_outlier = np.median(all_orders_with_outlier)

print("=== Impact d'UNE commande exceptionnelle sur avg_qty ===")
print(f"Commandes normales                : {normal_orders}")
print(f"+ une commande exceptionnelle     : {outlier_order}")
print(f"avg_qty SANS l'outlier            : {avg_without:.2f}")
print(f"avg_qty AVEC l'outlier            : {avg_with_outlier:.2f}  <- utilisé comme feature ET comme baseline")
print(f"median_qty AVEC l'outlier         : {median_with_outlier:.2f}  <- resterait proche du comportement normal")
print(f"max_qty AVEC l'outlier            : {max(all_orders_with_outlier)}  <- feature utilisée telle quelle")


=== Impact d'UNE commande exceptionnelle sur avg_qty ===
Commandes normales                : [4, 5, 3, 6, 4, 5]
+ une commande exceptionnelle     : [300]
avg_qty SANS l'outlier            : 4.50
avg_qty AVEC l'outlier            : 46.71  <- utilisé comme feature ET comme baseline
median_qty AVEC l'outlier         : 5.00  <- resterait proche du comportement normal
max_qty AVEC l'outlier            : 300  <- feature utilisée telle quelle


**Observation clé** : `avg_qty` passe de 4.5 à 46.7 à cause d'UNE SEULE commande passée. Or `avg_qty` est à la fois :
- une **feature d'entrée** du régresseur (le modèle apprend dessus)
- la **baseline de comparaison** (`evaluate_regressor` compare XGBoost contre cette moyenne polluée)

Le régresseur n'a aucun moyen de savoir que cette commande de 300 était une exception plutôt que la norme — statistiquement, elle FAIT PARTIE de l'historique "normal" de cette paire aux yeux du modèle.


### Cause n°2 — La fonction de perte quadratique amplifie l'effet des grosses commandes

`train_regressor.py` utilise `objective="reg:squarederror"`. Ça veut dire que l'algorithme essaie de minimiser la somme des **carrés** des erreurs. Une erreur de 300 unités compte 3600 fois plus qu'une erreur de 5 unités (300² / 5² = 3600). Concrètement, ça pousse le modèle à "prendre très au sérieux" les rares grosses commandes du dataset, bien plus proportionnellement que leur fréquence réelle ne le justifierait.


In [1]:
import numpy as np
np.random.seed(42)

small_orders = np.random.poisson(5, 200)          # commandes typiques
big_orders = np.array([250, 300, 180, 400, 220])   # rares commandes exceptionnelles

all_qty = np.concatenate([small_orders, big_orders])

print("=== Effet de la distribution asymétrique (skewed) sur l'apprentissage ===")
print(f"Nombre total de commandes         : {len(all_qty)}")
print(f"Commandes 'normales' (<20 unités) : {(all_qty < 20).sum()} ({(all_qty < 20).mean()*100:.1f}%)")
print(f"Commandes 'exceptionnelles' (>=20): {(all_qty >= 20).sum()} ({(all_qty >= 20).mean()*100:.1f}%)")
print(f"Moyenne brute                     : {all_qty.mean():.2f}")
print(f"Médiane                           : {np.median(all_qty):.2f}")
print()

sq_error_outliers = ((big_orders - all_qty.mean())**2).sum()
sq_error_normals  = ((small_orders - all_qty.mean())**2).sum()
print("=== Poids de l'erreur quadratique (ce que 'reg:squarederror' minimise) ===")
print(f"Erreur due aux {len(big_orders)} commandes exceptionnelles : {sq_error_outliers:,.0f}")
print(f"Erreur due aux {len(small_orders)} commandes normales      : {sq_error_normals:,.0f}")
print(f"-> {len(big_orders)} commandes rares pèsent {sq_error_outliers/sq_error_normals*100:.0f}% du poids "
      f"des 200 commandes normales dans la loss !")


=== Effet de la distribution asymétrique (skewed) sur l'apprentissage ===
Nombre total de commandes         : 205
Commandes 'normales' (<20 unités) : 200 (97.6%)
Commandes 'exceptionnelles' (>=20): 5 (2.4%)
Moyenne brute                     : 11.42
Médiane                           : 5.00

=== Poids de l'erreur quadratique (ce que 'reg:squarederror' minimise) ===
Erreur due aux 5 commandes exceptionnelles : 363,119
Erreur due aux 200 commandes normales      : 9,349
-> 5 commandes rares pèsent 3884% du poids des 200 commandes normales dans la loss !


**5 commandes exceptionnelles (2.4% du dataset) pèsent presque 39 fois plus que les 200 commandes normales** dans ce que le modèle essaie d'optimiser. Ce n'est pas une anomalie du code — c'est une conséquence mathématique directe du choix de `reg:squarederror` sur des données de vente, qui sont presque toujours asymétriques (beaucoup de petites commandes, quelques grosses).


### Cause n°3 — La parcimonie des données (sparsity) : peu d'observations par paire précise

Avec les vrais chiffres de ce projet (737 clients, 632 produits, 15 536 paires réellement réachetées), la matrice client×produit est extrêmement clairsemée.


In [1]:
n_clients = 737
n_products = 632
n_positive_pairs = 15536

print("=== Le problème de la parcimonie (sparsity) des données ===")
print(f"Clients uniques                        : {n_clients}")
print(f"Produits uniques                       : {n_products}")
print(f"Paires (client, produit) avec réachat  : {n_positive_pairs}")
print(f"Combinaisons théoriques possibles      : {n_clients * n_products:,}")
print(f"Taux de remplissage réel de la matrice : {n_positive_pairs/(n_clients*n_products)*100:.3f}%")
print()
print("Conséquence concrète :")
print("Pour un modèle précis de téléphone donné (ex: NOKIA 150 DS), la plupart")
print("des paires (client, ce produit) n'ont qu'1 ou 2 commandes dans tout l'historique.")
print("Une seule commande inhabituelle devient alors 100% de 'l'avg_qty' de cette paire —")
print("il n'y a aucune autre observation pour la contrebalancer.")


=== Le problème de la parcimonie (sparsity) des données ===
Clients uniques                        : 737
Produits uniques                       : 632
Paires (client, produit) avec réachat  : 15536
Combinaisons théoriques possibles      : 465,784
Taux de remplissage réel de la matrice : 3.335%

Conséquence concrète :
Pour un modèle précis de téléphone donné (ex: NOKIA 150 DS), la plupart
des paires (client, ce produit) n'ont qu'1 ou 2 commandes dans tout l'historique.
Une seule commande inhabituelle devient alors 100% de 'l'avg_qty' de cette paire —
il n'y a aucune autre observation pour la contrebalancer.


**Seulement 3.3% de la matrice client×produit est remplie.** Avec si peu de données par paire, il n'y a statistiquement aucune chance de "moyenner" un outlier — la commande exceptionnelle EST l'historique, aux yeux du modèle. Un arbre XGBoost avec `max_depth=6` et 500 arbres peut très bien isoler cette paire précise dans une feuille dédiée et reproduire fidèlement cette valeur exceptionnelle, ce qui ressemble à du "surapprentissage local" sur un point de donnée bruité.


### Cause n°4 — Aucun plafond de bon sens n'est appliqué en sortie

Regardons `recommendation.py`, la partie qui utilise le régresseur en production :

```python
regressor_qty = np.clip(np.round(raw_qty), 1, None).astype(int)
```

`np.clip(..., 1, None)` — le `None` en deuxième argument (le plafond max) signifie **littéralement aucune limite supérieure**. Le seul garde-fou est "au moins 1", jamais "au maximum X". Une prédiction de 300 traverse directement jusqu'à l'interface du commercial sans aucun filtre de plausibilité business.

Pire — dans `schemas.py` :

```python
quantite_max = max(sugg_qty * 3, 5)
```

Si `sugg_qty = 300`, alors `quantite_max = 900`. Le nombre absurde n'est pas juste affiché, il est **amplifié** dans la borne supérieure affichée au commercial.


### Cause n°5 — L'explication ne référence jamais le "pourquoi" du chiffre

Regardons `_rule_based_explanation` et le prompt envoyé au LLM dans `explanation.py`. Aucun des deux ne mentionne explicitement `max_qty`, ni ne compare `quantite_suggeree` à l'historique réel du client pour dire par exemple *"cette quantité est proche de votre plus grosse commande passée (300 unités le [date])"*.

Résultat : le commercial voit "300 unités (Confiance 99%)" sans AUCUNE traçabilité vers la donnée source qui justifie ce chiffre. Même si la prédiction avait une explication statistique légitime (peut-être ce client fait VRAIMENT des restocks ponctuels de 300), rien dans le texte ne le dit — d'où l'impression de résultat absurde et injustifié.


---
## 4. Synthèse — les 5 causes qui se combinent

```
1. avg_qty/max_qty non plafonnés  →  un seul outlier historique pollue la feature à vie
2. reg:squarederror                →  amplifie mathématiquement le poids des grosses commandes
3. Sparsity extrême (3.3%)         →  aucune donnée pour "diluer" un point aberrant
4. np.clip(..., 1, None)           →  aucun plafond business appliqué en sortie
5. Explications non traçables      →  le commercial ne peut jamais vérifier la logique
```

Aucune de ces causes n'est un "bug" au sens strict — chacune est une conséquence logique d'un choix de conception fait sans mauvaise intention. Mais ensemble, elles créent exactement le symptôme observé : une prédiction numériquement plausible pour le modèle, mais commercialement absurde et invérifiable pour l'humain qui la reçoit.


---
## 5. Pistes de correction (à prioriser ensemble, non codées ici)

| Piste | Ce que ça change | Effort |
|---|---|---|
| **Log-transform de la cible** : entraîner sur `log1p(target_qty)`, inverser avec `expm1()` à l'inférence | Compresse l'échelle des outliers, réduit leur poids disproportionné dans la loss quadratique | Faible — 2 lignes dans train_regressor.py |
| **Winsorization des features d'historique** | Plafonner `avg_qty`/`max_qty` au 95e ou 99e percentile pendant `feature_engineering.py` | Faible — ajout dans build_feature_matrix |
| **Plafond de bon sens à l'inférence** | `np.clip(raw_qty, 1, max_qty_historique * facteur)` dans recommendation.py | Faible — 1 ligne |
| **Perte robuste aux outliers** | Remplacer `reg:squarederror` par `reg:pseudohubererror` (moins sensible aux valeurs extrêmes) | Faible — 1 paramètre XGBoost |
| **Explication traçable** | Toujours inclure `avg_qty`, `max_qty`, `last_qty` dans le prompt LLM et le template de fallback, avec comparaison explicite | Moyen — modification du prompt |
| **Évaluation par quantile d'erreur** | Rapporter P50/P90/P99 de l'erreur, pas seulement le MAE moyen, pour détecter ce type de cas | Faible — ajout dans evaluate_regressor |

La correction la plus rapide à fort impact est la combinaison **log-transform + plafond de bon sens** — elle attaque à la fois la cause racine (comment le modèle apprend) et le symptôme final (ce qui sort réellement vers le commercial).
